![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 4 — Analítica, ML, IA y GenAI
**Rol:** CRB_ANALITICA | **Tiempo:** 15 min | **Criterio:** AI functions, agentes RAG, observabilidad, IA responsable

In [ ]:
USE ROLE CRB_ANALITICA;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Ejecutar: AI nativa en SQL (no notebooks)

Usamos **Cortex AI** para analizar documentos con lenguaje natural. El modelo LLM extrae campos estructurados desde texto libre — sin exportar datos a herramientas externas.

In [ ]:
-- AI_COMPLETE: extracción de campos desde texto no estructurado
SELECT documento_id,
  AI_COMPLETE('llama3.1-70b',
    'Del siguiente texto extrae: tipo de riesgo, entidad involucrada y monto. Responde en JSON: ' || TEXTO_DOCUMENTO
  )::VARCHAR AS extraccion
FROM CREDIBANCO_HOL.CUMPLIMIENTO.DOCUMENTOS_SARLAFT LIMIT 3;

**AI_CLASSIFY** categoriza alertas por severidad automáticamente. Snowflake clasifica texto directamente en SQL — sin pipelines de ML separados.

In [ ]:
-- AI_CLASSIFY: clasificación de severidad con labels cortos
SELECT alerta_id, comercio_id, descripcion,
  AI_CLASSIFY(descripcion, ARRAY_CONSTRUCT('ALT','MED','BAJ')):labels[0]::VARCHAR AS severidad
FROM CREDIBANCO_HOL.RIESGO.ALERTAS LIMIT 5;

**RAG nativo** sobre documentos SARLAFT. Cortex Search indexa y busca documentos de cumplimiento para responder preguntas regulatorias — sin vectorDB externo.

In [ ]:
-- Cortex Search: RAG sobre documentos SARLAFT (ya indexado)
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'CREDIBANCO_HOL.CUMPLIMIENTO.CSS_SARLAFT_DOCS_USER',
    '{"query": "lavado de activos", "columns": ["TEXTO_DOCUMENTO","TIPO_ALERTA"], "limit": 3}'
  )
);

## Bloque 2 — Evidencia: Anomalías y riesgo pre-computados

Las anomalías fueron detectadas con ML nativo de Snowflake. El modelo identifica comercios con comportamiento transaccional inusual — candidatos a investigación de fraude.

In [ ]:
-- Anomalías detectadas por comercio
SELECT * FROM CREDIBANCO_HOL.ANALITICA.RESULTADO_ANOMALIAS
WHERE IS_ANOMALY = TRUE LIMIT 10;

El scoring de riesgo por comercio combina features transaccionales con indicadores de fraude. Este dataset alimenta los modelos de ML del Track 4B (MLOps).

In [ ]:
-- Scoring de riesgo por comercio
SELECT ES_FRAUDE, COUNT(*) AS n, ROUND(AVG(TICKET_PROMEDIO),0) AS ticket_prom
FROM CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO
GROUP BY 1;

## Bloque 3 — Agente de IA + CoCo

### Crear un Agente de IA
Copia este prompt en **Cortex Code**:

> **Crea un Cortex Agent llamado AGENTE_FRAUDE_CREDIBANCO en CREDIBANCO_HOL.ANALITICA que use el Cortex Search Service CSS_SARLAFT_DOCS_USER para responder preguntas sobre documentos SARLAFT y riesgo de lavado de activos. Incluye 3 sample questions en español sobre alertas SARLAFT, comercios sospechosos y regulación antilavado. Usa el modelo llama3.1-70b. Pruébalo con una pregunta.**

El agente quedará disponible en **Snowflake Intelligence** para que cualquier usuario haga preguntas sobre cumplimiento sin acceder a la base de datos directamente.

In [ ]:
-- Verificación final
SELECT 'T4_COMPLETO' AS status,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.ANALITICA.RESULTADO_ANOMALIAS WHERE IS_ANOMALY = TRUE) AS anomalias;